# Classical ML Baseline

ECFP fingerprints + RDKit descriptors -> XGBoost / LightGBM / Random Forest,
evaluated on both the scaffold split (primary) and random split (contrast only).
Uncertainty via conformal prediction intervals (MAPIE). Applicability domain
via Tanimoto similarity to nearest training neighbor.

In [ ]:
import sys
import json
from pathlib import Path

import joblib
import pandas as pd
import wandb
import yaml
from sklearn.model_selection import GroupKFold, cross_validate
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.components.feature_engineering import featurize, murcko_scaffold
from src.components.model_evaluator import applicability_domain, regression_metrics
from src.components.model_trainer import (
    _tqdm_joblib,
    get_estimator,
    predict_with_interval,
    search_hyperparameters,
    train_with_uncertainty,
)

config = yaml.safe_load((ROOT / "configs" / "config.yaml").read_text())
SEED = config["seed"]
WANDB_PROJECT = config["wandb"]["project"]

In [ ]:
SPLITS_DIR = ROOT / config["data"]["splits_dir"]

def load_split(name, part):
    return pd.read_csv(SPLITS_DIR / f"{name}_{part}.csv")


splits = {
    name: {part: load_split(name, part) for part in ("train", "val", "test")}
    for name in ("scaffold", "random")
}
# Sanity check row counts per split before doing any modeling.
{name: {p: len(df) for p, df in parts.items()} for name, parts in splits.items()}

## Train grid

Model type x split type, ECFP fingerprints (the standard classical ML baseline
representation), default hyperparameters — no tuning yet. This picks the best
baseline model class before spending search budget on it. Each run is logged
to W&B with split type, descriptor type, hyperparameters, seed, dataset size,
and val/test metrics.

In [ ]:

MODEL_TYPES = ["xgboost", "lightgbm", "random_forest"]
DESCRIPTOR_KIND = "ecfp"
N_BITS = config["features"]["ecfp_n_bits"]

AD_THRESHOLD = config["applicability_domain"]["tanimoto_threshold"]

results = []
trained_models = {}


for split_name, parts in splits.items():
    Xtr = featurize(parts["train"], kind=DESCRIPTOR_KIND, n_bits=N_BITS)
    Xv = featurize(parts["val"], kind=DESCRIPTOR_KIND, n_bits=N_BITS)
    Xte = featurize(parts["test"], kind=DESCRIPTOR_KIND, n_bits=N_BITS)
    ytr, yv, yte = (parts[p]["pIC50"].values for p in ("train", "val", "test"))

    for model_type in MODEL_TYPES:
    
        wandb.init(
            project=WANDB_PROJECT,
            job_type="classical_ml",
            group=split_name,
            name=f"{model_type}_{split_name}",
            config={
                "model_type": model_type,
                "split_type": split_name,
                "descriptor_type": DESCRIPTOR_KIND,
                "seed": SEED,
                "n_train": len(Xtr),
                "n_val": len(Xv),
                "n_test": len(Xte),
            },
        )

        # Default hyperparameters at this stage: the goal here is choosing
        # the best model class, not the best hyperparameters yet.
        model = train_with_uncertainty(model_type, Xtr, ytr, Xv, yv, seed=SEED)

    
        val_pred, val_lo, val_hi = predict_with_interval(model, Xv)
        test_pred, test_lo, test_hi = predict_with_interval(model, Xte)
        val_metrics = regression_metrics(yv, val_pred)
        test_metrics = regression_metrics(yte, test_pred)

        # Applicability domain check: for each test molecule, find its
        # Tanimoto similarity to the nearest training set molecule.
        # Molecules below AD_THRESHOLD get flagged as out of domain.
        sim, in_domain = applicability_domain(
            parts["test"]["smiles"], parts["train"]["smiles"], threshold=AD_THRESHOLD
        )

        wandb.log({f"val_{k}": v for k, v in val_metrics.items()})
        wandb.log({f"test_{k}": v for k, v in test_metrics.items()})
        wandb.log(
            {
                "test_mean_interval_width": (test_hi - test_lo).mean(),
                "test_ad_in_domain_frac": in_domain.mean(),
            }
        )
        wandb.finish()

        trained_models[(model_type, split_name)] = model
        results.append({
            "model_type": model_type,
            "split_type": split_name,
            **{f"val_{k}": v for k, v in val_metrics.items()},
            **{f"test_{k}": v for k, v in test_metrics.items()},
            "test_mean_interval_width": (test_hi - test_lo).mean(),
            "test_ad_in_domain_frac": in_domain.mean(),
        })

results_df = pd.DataFrame(results)
results_df

## Results

Scaffold split is the headline result here, since it best represents
prospective use on new chemical series. Random split is shown only for
contrast: it typically looks better because train and test share scaffolds,
which inflates apparent generalization.

In [ ]:
# Sort within each split type by test RMSE.
results_df.sort_values(["split_type", "test_rmse"]).reset_index(drop=True)

In [ ]:
(ROOT / config["results_dir"]).mkdir(exist_ok=True)

results_df.to_csv(ROOT / config["results_dir"] / "classical_ml_results.csv", index=False)

gap = (
    results_df.pivot(index="model_type", columns="split_type", values="test_rmse")
    .assign(scaffold_minus_random=lambda d: d["scaffold"] - d["random"])
)
gap

## Tune the best baseline model

Pick the single best baseline model class (lowest scaffold split val RMSE —
scaffold is the primary evaluation split, so selection uses it, not the
optimistic random split).

In [ ]:
# Model class selection uses scaffold split validation RMSE, not random split.
baseline_scaffold = results_df[results_df["split_type"] == "scaffold"]
best_model_type = baseline_scaffold.sort_values("val_rmse").iloc[0]["model_type"]

parts = splits["scaffold"]
Xtr = featurize(parts["train"], kind=DESCRIPTOR_KIND, n_bits=N_BITS)
Xv = featurize(parts["val"], kind=DESCRIPTOR_KIND, n_bits=N_BITS)
Xte = featurize(parts["test"], kind=DESCRIPTOR_KIND, n_bits=N_BITS)
ytr, yv, yte = (parts[p]["pIC50"].values for p in ("train", "val", "test"))

# GroupKFold on Murcko scaffold, not plain KFold.
groups = parts["train"]["smiles"].apply(murcko_scaffold).values


cache_path = ROOT / "results" / "best_params_cache.json"
if cache_path.exists():
    cached = json.loads(cache_path.read_text())
    best_model_type, best_params, search_cv_rmse = (
        cached["best_model_type"],
        cached["best_params"],
        cached["search_cv_rmse"],
    )
else:
    best_params, search_cv_rmse = search_hyperparameters(
        best_model_type, Xtr, ytr, groups=groups, seed=SEED
    )
    cache_path.write_text(json.dumps({
        "best_model_type": best_model_type,
        "best_params": best_params,
        "search_cv_rmse": search_cv_rmse,
    }))
best_model_type, best_params, search_cv_rmse

## Validate the tuned hyperparameters with 5-fold CV

Separate from the search's internal CV score: refit the *tuned* estimator
across 5 scaffold grouped folds and check RMSE spread across folds. A wide
spread means the tuned hyperparameters are unstable / overfit to the
particular scaffold split — a narrow spread confirms they generalize before
committing to a final fit.

In [ ]:
cv = GroupKFold(n_splits=5)
with _tqdm_joblib(cv.get_n_splits(groups=groups), desc="5-fold CV"):
    cv_scores = cross_validate(
        get_estimator(best_model_type, SEED, **best_params),
        Xtr,
        ytr,
        groups=groups,
        cv=cv,

        scoring=["neg_root_mean_squared_error", "neg_mean_absolute_error", "r2"],
        n_jobs=-1,
    )

cv_summary = {
    "cv_rmse_mean": -cv_scores["test_neg_root_mean_squared_error"].mean(),
    "cv_rmse_std": (-cv_scores["test_neg_root_mean_squared_error"]).std(),
    "cv_mae_mean": -cv_scores["test_neg_mean_absolute_error"].mean(),
    "cv_r2_mean": cv_scores["test_r2"].mean(),
}
cv_summary

## Final fit & register on W&B

Fit the tuned model on the full scaffold train set, conformalize on val
(needed for prediction intervals — must stay held out from the CV above),
evaluate on test, and register as the W&B model registry artifact for this
model class.

In [ ]:
final_model = train_with_uncertainty(
    best_model_type, Xtr, ytr, Xv, yv, seed=SEED, **best_params
)

val_pred, val_lo, val_hi = predict_with_interval(final_model, Xv)
test_pred, test_lo, test_hi = predict_with_interval(final_model, Xte)
val_metrics = regression_metrics(yv, val_pred)
test_metrics = regression_metrics(yte, test_pred)

sim, in_domain = applicability_domain(
    parts["test"]["smiles"], parts["train"]["smiles"], threshold=AD_THRESHOLD
)

(ROOT / config["models_dir"]).mkdir(exist_ok=True)
path = ROOT / config["models_dir"] / f"{best_model_type}_scaffold_tuned.joblib"
joblib.dump(final_model, path)

run = wandb.init(
    project=WANDB_PROJECT,
    job_type="classical_ml_tuned",
    group="scaffold",
    name=f"{best_model_type}_scaffold_tuned",
    config={
        "model_type": best_model_type,
        "split_type": "scaffold",
        "descriptor_type": DESCRIPTOR_KIND,
        "seed": SEED,
        "n_train": len(Xtr),
        "n_val": len(Xv),
        "n_test": len(Xte),
        "cv_folds": 5,
        "cv_grouping": "murcko_scaffold",
        **{f"best_{k}": v for k, v in best_params.items()},
    },
)
wandb.log({"search_cv_rmse": search_cv_rmse, **cv_summary})
wandb.log({f"val_{k}": v for k, v in val_metrics.items()})
wandb.log({f"test_{k}": v for k, v in test_metrics.items()})
wandb.log(
    {
        "test_mean_interval_width": (test_hi - test_lo).mean(),
        "test_ad_in_domain_frac": in_domain.mean(),
    }
)

artifact = wandb.Artifact(
    f"{best_model_type}-scaffold-tuned",
    type="model",
    metadata={"best_params": best_params, **cv_summary},
)
artifact.add_file(str(path))
logged = run.log_artifact(artifact)
run.link_artifact(
    logged, target_path=f"wandb-registry-model/egfr-pic50-{best_model_type}"
)
wandb.finish()

test_metrics

## Overfit / underfit check


In [ ]:
# Only val/test performance was computed above. Add train set performance
# back in for comparison: a big gap between train and val/test, combined
# with near perfect train R2, indicates overfitting; uniformly low scores
# on all three indicate underfitting instead.
train_pred, *_ = predict_with_interval(final_model, Xtr)
train_metrics = regression_metrics(ytr, train_pred)

fit_diagnostic = pd.DataFrame(
    {"train": train_metrics, "val": val_metrics, "test": test_metrics}
).T[["rmse", "mae", "r2", "spearman"]]
print(f"train-val R2 gap: {train_metrics['r2'] - val_metrics['r2']:.3f}")
print("large gap + near-perfect train R2 -> overfit; both low -> underfit")
fit_diagnostic